# Time Series Feature Builder

**Level:** Advanced

## Project description

You are preparing daily sales data for forecasting. The model team needs
complete daily records, lag features, and rolling averages.

## Skills practiced

- Date parsing
- Resampling
- Filling missing dates
- Lag and rolling-window features


In [ ]:
import pandas as pd
import numpy as np
import secrets

practice_run_name = f"time-series-{secrets.token_hex(3)}"
print(f"Practice run: {practice_run_name}")

def make_daily_sales_data(day_count=45, missing_days=6, seed=42):
    """Create synthetic daily revenue with trend, weekly seasonality, and missing dates."""
    rng = np.random.default_rng(seed)
    dates = pd.date_range("2026-05-01", periods=day_count, freq="D")
    day_number = np.arange(day_count)
    weekly_pattern = np.where(dates.dayofweek >= 5, 180, 0)
    revenue = 1100 + (day_number * 18) + weekly_pattern + rng.normal(0, 120, size=day_count)

    sales = pd.DataFrame({
        "date": dates,
        "revenue": revenue.round(0).astype(int),
    })
    drop_count = min(missing_days, max(day_count - 1, 0))
    missing_index = rng.choice(sales.index, size=drop_count, replace=False)
    return sales.drop(index=missing_index).sort_values("date").reset_index(drop=True)

sales = make_daily_sales_data(day_count=45, missing_days=6, seed=42)
sales


## Step 1: Set a daily date index

Forecasting data usually needs one row per time period. Missing dates can
confuse models and charts, so we create a complete daily index.


In [ ]:
daily = (
    sales
    .set_index("date")
    .asfreq("D")
)

daily


## Step 2: Fill missing revenue values

For this training example, missing dates mean no recorded sales, so we fill
missing revenue with 0. In real work, always confirm this assumption.


In [ ]:
daily["revenue"] = daily["revenue"].fillna(0)
daily


## Step 3: Create lag features


In [ ]:
daily["revenue_lag_1"] = daily["revenue"].shift(1)
daily["revenue_lag_7"] = daily["revenue"].shift(7)

daily


## Step 4: Create rolling averages


In [ ]:
daily["rolling_3_day_revenue"] = daily["revenue"].rolling(window=3).mean()
daily["rolling_7_day_revenue"] = daily["revenue"].rolling(window=7).mean()

daily


## Step 5: Export the modeling table


In [ ]:
modeling_table = daily.reset_index()
modeling_table


## Final project notes

- Dates that were missing from the original data:
- Why filling missing revenue with 0 may or may not be correct:
- Most useful forecasting feature:
- One validation check you would add:


## Practice extension: Generate your own dataset

Change `day_count`, `missing_days`, and `seed` to create a new forecasting
practice dataset. Set `seed=None` for a randomized dataset each run. More
missing days will make the resampling step easier to see.


In [ ]:
my_sales = make_daily_sales_data(day_count=90, missing_days=14, seed=5)
my_sales.head()
